# Data


In [1]:
import os
import cv2 as c
import torch as t
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from collections import Counter

comp_train    = "/kaggle/input/competitions/emotion-detection-competition/Training_data/Training_data"
comp_test     = "/kaggle/input/competitions/emotion-detection-competition/test/test"
ferplus_train = "/kaggle/input/datasets/arnabkumarroy02/ferplus/train"
ferplus_val   = "/kaggle/input/datasets/arnabkumarroy02/ferplus/validation"


device = t.device("cuda" if t.cuda.is_available() else "cpu")

IMG_SIZE      = 128
VALID_CLASSES = {"fear", "happy", "disgust", "sad", "surprise", "angry"}
label_map     = {cls: i for i, cls in enumerate(sorted(VALID_CLASSES))}
idx_to_label  = {v: k for k, v in label_map.items()}

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomCrop(IMG_SIZE, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def load_images(paths, filter_classes=None):
    images, labels = [], []
    for path in paths:
        for cls_name in os.listdir(path):
            cls_normalized = "surprise" if cls_name == "suprise" else cls_name
            if filter_classes and cls_normalized not in filter_classes:
                continue
            cls_path = os.path.join(path, cls_name)
            if not os.path.isdir(cls_path):
                continue
            for img_file in os.listdir(cls_path):
                if not img_file.endswith(('.jpg', '.png')):
                    continue
                img = c.imread(os.path.join(cls_path, img_file), c.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                images.append(img)
                labels.append(label_map[cls_normalized])
    return images, labels

def load_test_images(path):
    images, filenames = [], []
    for img_file in os.listdir(path):
        if not img_file.endswith('.jpg'):
            continue
        img = c.imread(os.path.join(path, img_file), c.IMREAD_GRAYSCALE)
        if img is None:
            continue
        images.append(img)
        filenames.append(img_file)
    return images, filenames

class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, t.tensor(label, dtype=t.long)

class TestDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images    = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img

print("Loading training data...")
train_imgs, train_lbls = load_images(
    paths=[comp_train, ferplus_train, ferplus_val],
    filter_classes=VALID_CLASSES
)

print("Loading test data...")
test_imgs, test_filenames = load_test_images(comp_test)

print(f"Total train samples : {len(train_imgs)}")
print(f"Class distribution  : {Counter(train_lbls)}")
print(f"Total test samples  : {len(test_imgs)}")
print(f"Label map           : {label_map}")

class_counts  = Counter(train_lbls)
total_samples = len(train_lbls)
class_weights = t.tensor([
    total_samples / (len(class_counts) * class_counts[i])
    for i in range(len(label_map))
], dtype=t.float32).to(device)

print(f"Class weights       : {class_weights}")

full_dataset = EmotionDataset(train_imgs, train_lbls, transform=train_transform)
val_size     = int(0.15 * len(full_dataset))
train_size   = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                 generator=t.Generator().manual_seed(42))

# val بياخد test_transform مش train_transform
val_ds = EmotionDataset(
    [train_imgs[i] for i in val_ds.indices],
    [train_lbls[i] for i in val_ds.indices],
    transform=test_transform
)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(TestDataset(test_imgs, transform=test_transform), batch_size=32)

print(f"\nTrain batches : {len(train_dl)}")
print(f"Val batches   : {len(val_dl)}")
print(f"Test batches  : {len(test_dl)}")

Loading training data...
Loading test data...
Total train samples : 87237
Class distribution  : Counter({3: 18690, 5: 14032, 4: 14015, 2: 13500, 0: 13500, 1: 13500})
Total test samples  : 7116
Label map           : {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4, 'surprise': 5}
Class weights       : tensor([1.0770, 1.0770, 1.0770, 0.7779, 1.0374, 1.0362], device='cuda:0')

Train batches : 2318
Val batches   : 409
Test batches  : 223


# model

In [2]:
class VGGNet(nn.Module):
    def __init__(self, num_classes=6):
        super(VGGNet, self).__init__()
        self.features = nn.Sequential(
            # Block 1: (1, 128, 128) → (64, 64, 64)
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
            # Block 2: (64, 64, 64) → (128, 32, 32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
            # Block 3: (128, 32, 32) → (256, 16, 16)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
            # Block 4: (256, 16, 16) → (512, 8, 8)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
            # Block 5: (512, 8, 8) → (512, 4, 4)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Kaiming initialization
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)

model = VGGNet(num_classes=6).to(device)
model.apply(init_weights)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Total parameters: 14,854,854


# training 


In [3]:
import pandas as pd
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = t.compile(model)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# scaler بره الـ loop مش جواه
scaler = t.cuda.amp.GradScaler()

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with t.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with t.cuda.amp.autocast():
                outputs = model(imgs)
                loss    = criterion(outputs, lbls)
            total_loss += loss.item() * imgs.size(0)
            correct    += outputs.argmax(dim=1).eq(lbls).sum().item()
            total      += lbls.size(0)
    return total_loss / total, 100 * correct / total

def check_overfitting(train_acc, val_acc, train_loss, val_loss, gap_threshold=10.0):
    gap = train_acc - val_acc
    print(f"  ↳ Acc Gap (train - val): {gap:.2f}%", end="  ")
    if gap > gap_threshold:
        print(f"⚠️  Overfitting! gap > {gap_threshold}%")
    elif val_loss > train_loss * 1.3:
        print("⚠️  Overfitting! val_loss >> train_loss")
    else:
        print("✅ OK")

EPOCHS           = 60
PATIENCE         = 10
best_val_acc     = 0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for imgs, lbls in train_dl:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()

        with t.cuda.amp.autocast():
            outputs = model(imgs)
            loss    = criterion(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        correct    += outputs.argmax(dim=1).eq(lbls).sum().item()
        total      += lbls.size(0)

    train_loss_avg    = total_loss / len(train_dl)
    train_acc         = 100 * correct / total
    val_loss, val_acc = evaluate(model, val_dl)

    scheduler.step(val_acc)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}]  "
          f"Train Loss: {train_loss_avg:.4f}  Train Acc: {train_acc:.2f}%  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    check_overfitting(train_acc, val_acc, train_loss_avg, val_loss)

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        t.save(model.state_dict(), "best_vgg.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stop  |  best val acc: {best_val_acc:.2f}%")
            break

print(f"\nBest Val Acc: {best_val_acc:.2f}%")

/tmp/ipykernel_23/3529301635.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = t.cuda.amp.GradScaler()
/tmp/ipykernel_23/3529301635.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t.cuda.amp.autocast():
W0519 18:20:05.521000 23 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/tmp/ipykernel_23/3529301635.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t.cuda.amp.autocast():
/tmp/ipykernel_23/3529301635.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t.cuda.amp.autocast():
/tmp/ipykernel_23/3529301635.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autoc

Epoch [01/60]  Train Loss: 1.6844  Train Acc: 29.34%  Val Loss: 1.3718  Val Acc: 50.32%
  ↳ Acc Gap (train - val): -20.98%  ✅ OK


/tmp/ipykernel_23/3529301635.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t.cuda.amp.autocast():


Epoch [02/60]  Train Loss: 1.2666  Train Acc: 55.24%  Val Loss: 1.0145  Val Acc: 69.06%
  ↳ Acc Gap (train - val): -13.82%  ✅ OK
Epoch [03/60]  Train Loss: 1.0597  Train Acc: 65.76%  Val Loss: 0.8980  Val Acc: 73.34%
  ↳ Acc Gap (train - val): -7.57%  ✅ OK
Epoch [04/60]  Train Loss: 0.9573  Train Acc: 70.59%  Val Loss: 0.8119  Val Acc: 76.62%
  ↳ Acc Gap (train - val): -6.03%  ✅ OK
Epoch [05/60]  Train Loss: 0.8887  Train Acc: 73.86%  Val Loss: 0.7803  Val Acc: 78.02%
  ↳ Acc Gap (train - val): -4.16%  ✅ OK
Epoch [06/60]  Train Loss: 0.8393  Train Acc: 76.07%  Val Loss: 0.7540  Val Acc: 79.58%
  ↳ Acc Gap (train - val): -3.51%  ✅ OK
Epoch [07/60]  Train Loss: 0.7965  Train Acc: 77.95%  Val Loss: 0.6999  Val Acc: 81.86%
  ↳ Acc Gap (train - val): -3.91%  ✅ OK
Epoch [08/60]  Train Loss: 0.7652  Train Acc: 79.34%  Val Loss: 0.6992  Val Acc: 82.06%
  ↳ Acc Gap (train - val): -2.71%  ✅ OK
Epoch [09/60]  Train Loss: 0.7395  Train Acc: 80.43%  Val Loss: 0.6483  Val Acc: 84.39%
  ↳ Acc Gap (tr

# inference 

In [ ]:
import re
import torchvision.transforms as transforms

model.load_state_dict(t.load("best_vgg.pth"))

tta_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

TTA_RUNS = 5
model.eval()

all_preds = []
with t.no_grad():
    for imgs in test_dl:
        imgs = imgs.to(device)
        probs = t.zeros(imgs.size(0), 6).to(device)
        for _ in range(TTA_RUNS):
            with t.cuda.amp.autocast():
                probs += t.softmax(model(imgs), dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().tolist())

def filename_to_id(path):
    """Extract 'test (2)' from 'some/dir/test (2).jpg'"""
    match = re.search(r'([^/\\]+)(?=\.\w+$)', path)
    return match.group(1) if match else path

ids = [filename_to_id(f) for f in test_filenames]

df = pd.DataFrame({
    "ID": ids,
    "Label": [idx_to_label[p] for p in all_preds]
})

df.to_csv("submission.csv", index=False)
print(df.head(10))
print(f"\nSaved submission.csv → {len(df)} rows")
print(f"Best Val Acc: {best_val_acc:.2f}%")

/tmp/ipykernel_23/4266737841.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t.cuda.amp.autocast():


                ID     Label
0  test (3436).jpg     happy
1  test (1194).jpg       sad
2   test (744).jpg  surprise
3  test (5072).jpg     angry
4  test (4037).jpg     happy
5  test (4950).jpg   disgust
6  test (4913).jpg     angry
7  test (6461).jpg   disgust
8  test (6296).jpg   disgust
9   test (521).jpg       sad

Saved submission.csv → 7116 rows
Best Val Acc: 91.42%
